# Data Preparation Lab -- Module 2, Class 2

**Dataset:** Superstore Sales

In this lab you will:
1. Load and inspect data
2. Handle missing values
3. Remove duplicates
4. Convert data types
5. Create derived features

The first 3 tasks are pre-built. The rest are TODO for you.

---
## Setup: Load the Dataset

Option A: Upload from Kaggle (download from https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

Option B: Use the URL loader below.

In [ ]:
import pandas as pd
import numpy as np

# Option A: Upload in Colab
# from google.colab import files
# uploaded = files.upload()  # upload your CSV
# df = pd.read_csv('SampleSuperstore.csv')

# Option B: Load from a public URL
# If the URL does not work, use Option A.
url = "https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv"

try:
    df = pd.read_csv(url, encoding='latin-1')
    print(f"Loaded from URL: {df.shape[0]} rows, {df.shape[1]} columns")
except Exception as e:
    print(f"URL failed ({e}). Use Option A: upload the CSV manually.")
    # Fallback: upload manually
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename, encoding='latin-1')
    print(f"Loaded from upload: {df.shape[0]} rows, {df.shape[1]} columns")

# This code acts like a "Smart Mailman" that tries to grab your store data from the internet first.
# If the internet link is broken, it uses a backup plan that lets you pick the file from your own computer.
# It makes sure your information is ready in a digital table called 'df' so you can start cleaning it.

Loaded from URL: 10800 rows, 21 columns


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vivek468/superstore-dataset-final")

print("Path to dataset files:", path)

# This code is like a "Special Delivery" bot that goes to a giant toy warehouse called Kaggle to find the newest box of store data.
# It downloads the box to a secret spot on your computer and then tells you exactly where it hid it so you can go find it.

Using Colab cache for faster access to the 'superstore-dataset-final' dataset.
Path to dataset files: /kaggle/input/superstore-dataset-final


---
## Task 1: Inspect the Data (pre-built)

Always look at your data before doing anything to it.

In [ ]:
# First 5 rows
df.head(5)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164


In [ ]:
# Shape: rows x columns
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

# This code is like using a giant ruler to measure how big your table of data is.
# It tells you exactly how many rows and columns you have so you know how much information you're working with.

In [ ]:
# Data types and non-null counts
df.info()

# This code is like a health check-up for your data table.
# It tells you what kind of information is in each column (like numbers or names) and if there are any empty spots that need fixing.

In [ ]:
# Summary statistics for numerical columns
df.describe()

# This code is like a "Magic Math Report" that looks at all the numbers in your box.
# It quickly tells you the average, the smallest number, and the biggest number so you can see how much things usually cost.

---
## Task 2: Missing Values (pre-built)

Check which columns have missing values and how many.

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})

# Show only columns with missing values
missing_report[missing_report['missing_count'] > 0]

# This code is like a detective hunting for missing puzzle pieces in your big box of data.
# It counts every empty spot and shows you a list of only the parts that are broken so you know exactly what needs to be fixed.

In [ ]:
# Fill missing numerical values with median (robust to outliers)
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled {col} missing values with median: {median_val}")

# Fill missing categorical values with mode
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"Filled {col} missing values with mode: {mode_val}")

# Verify: no more missing values
print(f"\nTotal missing values remaining: {df.isnull().sum().sum()}")

# This code is like a "Fix-It Bot" that finds empty holes in your data and fills them in so the table is complete.
# It uses a "middle-sized" number to fix number holes and the most popular word to fix name holes.
# At the end, it double-checks to make sure every single gap is filled.

---
## Task 3: Remove Duplicates (pre-built)

In [ ]:
# Check for duplicates
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {n_duplicates}")

# Remove duplicates
if n_duplicates > 0:
    df = df.drop_duplicates()
    print(f"After removal: {df.shape[0]} rows remain")
else:
    print("No duplicates to remove.")

# This code is like a "Twin Detector" that hunts for identical copies of the same receipt.
# If it finds any, it keeps one and puts the extras in the recycling bin so you don't count the same sale twice.

---
## Task 4: Convert Date Columns (TODO)

The `Order Date` and `Ship Date` columns are stored as strings. Convert them to proper datetime objects.

Hint: Use `pd.to_datetime()`. After conversion, verify with `.dtypes`.

In [ ]:
# Check current types of date columns
print("Before conversion:")
for col in df.columns:
    if 'date' in col.lower() or 'Date' in col:
        print(f"  {col}: {df[col].dtype}")
        print(f"  Sample value: {df[col].iloc[0]}")

# This code is like a detective searching for any folder labeled 'Date' to see what's inside.
# It reports back what kind of information is in those folders and shows you a tiny sample so you can see if they look like real calendar dates or just plain words.

In [ ]:
# TODO: Convert date columns to datetime
# Look at the column names printed above and convert them.
# The column names may vary depending on the dataset version.
#
# Example pattern:
# df['Column Name'] = pd.to_datetime(df['Column Name'])

# Your code here:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# This code is like teaching the computer how to read a calendar.
# It turns plain words into real dates so the computer knows exactly when things happened and can put them in the right order.


In [ ]:
# TODO: Verify the conversion worked
# Print dtypes for the date columns to confirm they are datetime64

# Your code here:
print(df[['Order Date', 'Ship Date']].dtypes)

# This code is like checking your work after finishing a puzzle.
# It asks the computer to show you the labels on those columns one last time, just to be 100% sure it really understands they are dates now!


---
## Task 5: Derived Features (TODO)

Create customer-level summary features. These are the building blocks for customer segmentation (Activity 4).

You need to create:
- **total_spending**: Total sales per customer
- **order_frequency**: Number of orders per customer
- **avg_order_value**: Average sales amount per order per customer

Hint: Use `df.groupby('Customer ID')` (or whatever the customer ID column is named).

In [ ]:
# First, identify the right column names
print("All columns:")
for col in df.columns:
    print(f"  {col}")

# This code is like reading the table of contents for your big book of data.
# It prints out the name of every single column so you can find the exact "labels" needed to build your summary reports.

In [ ]:
# TODO: Create total_spending per customer
# Hint: df.groupby('Customer ID')['Sales'].sum()
#
# Replace column names below with the actual names from your dataset.

# customer_spending = df.groupby('???')['???'].sum()
# customer_spending.name = 'total_spending'

# Your code here:


In [ ]:
# TODO: Create order_frequency per customer
# Hint: Count the number of rows (orders) per customer.
# Use .groupby(...).size() or .groupby(...)['some_col'].count()

# Your code here:


In [ ]:
# TODO: Create avg_order_value per customer
# Hint: Use .groupby(...)['Sales'].mean()

# Your code here:


In [ ]:
# TODO: Combine all three into a single customer-level DataFrame
# Hint: Use pd.concat([series1, series2, series3], axis=1)
# or create them all at once with .groupby(...).agg(...)

# customer_summary = pd.concat([customer_spending, order_freq, avg_order], axis=1)
# customer_summary.columns = ['total_spending', 'order_frequency', 'avg_order_value']

# Your code here:


In [ ]:
# TODO: Display the first 10 rows of your customer summary and .describe()

# Your code here:


---
## Task 6: Save Cleaned Data (TODO)

Save the cleaned DataFrame to a new CSV file. Never overwrite the original.

In [ ]:
# TODO: Save the cleaned main DataFrame
# df.to_csv('superstore_cleaned.csv', index=False)

# TODO: Save the customer summary DataFrame
# customer_summary.to_csv('customer_summary.csv')

# Your code here:


---
## Reflection

Answer these in a text cell or comments:

1. Why did we use median instead of mean for filling numerical missing values?
2. What is the difference between the row-level DataFrame (one row per order) and the customer-level summary? When would you use each?
3. If two rows have identical values in every column, are they always true duplicates? When might they not be?

1. Why use median instead of mean?
The median is used because it isn't easily swayed by "outliers," which are numbers that are much higher or lower than the rest. It provides a more "normal" middle value that accurately represents the data even if there are a few extreme values.

2. Row-level vs. Customer-level?
A row-level table shows every single receipt or action, while a customer-level table is a "report card" that summarizes everything one person did. You use row-level data to see daily details and customer-level data to see who your most loyal shoppers are.

3. Are identical rows always duplicates?
Not necessarily, because someone could have simply bought the exact same item at the same price and time. However, in many datasets, this is often an error where the same information was accidentally recorded twice.